# Heart-Sound Classification: Class-Imbalance Mitigation and Cross-Database GeneralizationReproducible pipeline accompanying the manuscript submitted to the *American Journal of Student Research*.This notebook contains the final analysis pipeline in run order:1. Rehydrate (mount Drive, extract dataset, rebuild dataframe, restore spectrograms)2. Build the master dataframe and verify the PCG channel3. Generate Mel-spectrograms (explicit STFT parameters)4. Source-held-out (leave-one-database-out) training — folds a, b, e5. Within-distribution (pooled random split) training6. Confidence intervals and pairwise McNemar tests7. Training/validation curves (subset) and plotting**Dataset:** PhysioNet/CinC Challenge 2016 (v1.0.0), training folders a–f (3,240 recordings; 2,575 Normal, 665 Abnormal). The dataset is not redistributed here; download it from https://physionet.org/content/challenge-2016/1.0.0/ and place it in Drive as referenced below.**Determinism:** fixed seeds for Python/NumPy/PyTorch, deterministic cuDNN, seeded data-loader workers.**Hardware:** NVIDIA Tesla T4 (Google Colab). See `requirements.txt` for package versions.

## 1. Rehydrate — mount Drive, extract dataset, rebuild dataframe, restore spectrograms

In [ ]:
import os, glob, zipfile, shutil, pandas as pd
from google.colab import drive

drive.mount('/content/drive')

# --- Dataset (download from PhysioNet and place this zip in your Drive) ---
zip_drive = "/content/drive/MyDrive/ISEF_Project/classification-of-heart-sound-recordings-the-physionetcomputing-in-cardiology-challenge-2016-1.0.0.zip"
if not os.path.exists("/content/heart_data"):
    shutil.copy(zip_drive, "/content/heart.zip")
    with zipfile.ZipFile("/content/heart.zip") as z:
        z.extractall("/content/heart_data")
    print("Dataset extracted")
else:
    print("Dataset already present")

root = None
for dp, dn, fn in os.walk("/content/heart_data"):
    if any(d.startswith("training-") for d in dn):
        root = dp; break
print("Dataset root:", root)

## 2. Build the master dataframe and verify the PCG channelLabels: PhysioNet encodes -1 = Normal, 1 = Abnormal (mapped here to 0 = Normal, 1 = Abnormal).

In [ ]:
import wave, contextlib

records = []
for folder in sorted(glob.glob(os.path.join(root, "training-*"))):
    src = os.path.basename(folder)
    ref = pd.read_csv(os.path.join(folder, "REFERENCE.csv"), header=None, names=["recording_id","orig_label"])
    for _, r in ref.iterrows():
        rid = str(r["recording_id"]).strip()
        wav = os.path.join(folder, rid + ".wav")
        if os.path.exists(wav):
            label = 0 if int(r["orig_label"]) == -1 else 1
            records.append({"recording_id": rid, "wav_path": wav, "label": label, "source": src})

master_df = pd.DataFrame(records)
master_df.to_csv("/content/master_df.csv", index=False)
print("Total recordings:", len(master_df))
print(master_df.groupby(["source","label"]).size().unstack(fill_value=0))

# Verify .wav files are single-channel PCG at 2000 Hz (ECG lives separately in .dat and is never loaded)
def wav_channels(path):
    with contextlib.closing(wave.open(path,'rb')) as w:
        return w.getnchannels(), w.getframerate()
for src in sorted(master_df['source'].unique()):
    s = master_df[master_df.source==src]['wav_path'].iloc[0]
    print(src, wav_channels(s))

## 3. Generate Mel-spectrograms (explicit STFT parameters)2000 Hz, 4-second window (pad/truncate), n_fft=256, hop=64, Hann window, 64 Mel bands, fmax=1000 Hz, per-recording dB (ref=max), min–max normalized to [0,255] grayscale.

In [ ]:
import numpy as np, librosa
from PIL import Image
from tqdm import tqdm

SAMPLE_RATE=2000; DURATION=4; TARGET_LEN=SAMPLE_RATE*DURATION
N_FFT=256; WIN_LENGTH=256; HOP_LENGTH=64; WINDOW='hann'
N_MELS=64; FMAX=1000; CENTER=True
SPEC_DIR='/content/spectrograms_flat'; os.makedirs(SPEC_DIR, exist_ok=True)

def make_spectrogram(wav_path, out_path):
    y, sr = librosa.load(wav_path, sr=SAMPLE_RATE, duration=DURATION)
    if len(y) < TARGET_LEN: y = np.pad(y, (0, TARGET_LEN-len(y)))
    else: y = y[:TARGET_LEN]
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH, window=WINDOW, center=CENTER, n_mels=N_MELS, fmax=FMAX, power=2.0)
    S_db = librosa.power_to_db(S, ref=np.max)
    lo, hi = S_db.min(), S_db.max()
    S_norm = (S_db-lo)/(hi-lo) if hi>lo else np.zeros_like(S_db)
    Image.fromarray((S_norm*255).astype(np.uint8)).save(out_path)

# Skip if already restored from backup zip
existing = glob.glob(f'{SPEC_DIR}/*.png')
if len(existing) < len(master_df):
    for _, r in tqdm(master_df.iterrows(), total=len(master_df)):
        make_spectrogram(r['wav_path'], os.path.join(SPEC_DIR, r['recording_id']+'.png'))
print('Spectrograms on disk:', len(glob.glob(f'{SPEC_DIR}/*.png')))

## Shared training machineryModel, dataset, deterministic seeding, and the training/evaluation functions used by all experiments.

In [ ]:
import random, json
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SPEC_DIR = '/content/spectrograms_flat'
OUT_DIR = '/content/drive/MyDrive/ISEF_Project/results_final'; os.makedirs(OUT_DIR, exist_ok=True)
EPOCHS=15; BATCH=32; LR=0.001; MOMENTUM=0.9
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

tfm = transforms.Compose([transforms.Resize((224,224)), transforms.Grayscale(3),
    transforms.ToTensor(), transforms.Normalize(MEAN,STD)])

class SpecDS(Dataset):
    def __init__(self, df): self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(os.path.join(SPEC_DIR, r['recording_id']+'.png')).convert('L')
        return tfm(img), int(r['label']), r['recording_id']

def build_model():
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(DEVICE)

def evaluate(model, loader):
    model.eval(); ys, ps, ids = [], [], []
    with torch.no_grad():
        for x, y, rid in loader:
            pred = model(x.to(DEVICE)).argmax(1).cpu().numpy()
            ys += list(y.numpy()); ps += list(pred); ids += list(rid)
    ys, ps = np.array(ys), np.array(ps)
    tn,fp,fn,tp = confusion_matrix(ys,ps,labels=[0,1]).ravel(); eps=1e-9
    return {'accuracy':(tp+tn)/(tp+tn+fp+fn),'sensitivity':tp/(tp+fn+eps),
            'specificity':tn/(tn+fp+eps),'precision':tp/(tp+fp+eps),
            'f1':2*tp/(2*tp+fp+fn+eps),'tp':int(tp),'tn':int(tn),'fp':int(fp),'fn':int(fn)}, \
           list(zip(ids, ys.tolist(), ps.tolist()))

def train_one(config, seed, tr, va, te):
    set_seed(seed); g = torch.Generator(); g.manual_seed(seed)
    if config=='oversampled':
        counts = tr['label'].value_counts().to_dict()
        w = tr['label'].map(lambda c: 1.0/counts[c]).values
        sampler = WeightedRandomSampler(torch.DoubleTensor(w), len(w), True, generator=g)
        trl = DataLoader(SpecDS(tr), BATCH, sampler=sampler)
    else:
        trl = DataLoader(SpecDS(tr), BATCH, shuffle=True, generator=g)
    val = DataLoader(SpecDS(va), BATCH); tel = DataLoader(SpecDS(te), BATCH)
    if config=='class_weighted':
        n = tr['label'].value_counts().to_dict()
        cw = torch.tensor([1.0/n[0],1.0/n[1]],dtype=torch.float).to(DEVICE); cw = cw/cw.sum()*2
        criterion = nn.CrossEntropyLoss(weight=cw)
    else:
        criterion = nn.CrossEntropyLoss()
    model = build_model(); opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM)
    best_f1, best_state = -1, None
    for ep in range(EPOCHS):
        model.train()
        for x,y,_ in trl:
            opt.zero_grad(); criterion(model(x.to(DEVICE)), y.to(DEVICE)).backward(); opt.step()
        vm,_ = evaluate(model, val)
        if vm['f1']>best_f1: best_f1=vm['f1']; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best_state)
    tm, preds = evaluate(model, tel); tm['best_val_f1']=best_f1
    return tm, preds

## 4. Source-held-out (leave-one-database-out) training — folds a, b, eHold out one large database at a time; c/d/f always remain in training. 3 models × 3 seeds per fold.

In [ ]:
SEEDS=[0,1,2]
HELD_OUT_FOLDS=['training-a','training-b','training-e']

for HOLD in HELD_OUT_FOLDS:
    test_df = master_df[master_df['source']==HOLD].copy()
    pool = master_df[master_df['source']!=HOLD].copy()
    tr, va = train_test_split(pool, test_size=0.15, stratify=pool['label'], random_state=42)
    n_ab=int((test_df.label==1).sum()); n_no=int((test_df.label==0).sum())
    print(f'\nHOLD OUT {HOLD} | Train {len(tr)} Val {len(va)} Test {len(test_df)} '
          f'| majority-acc={max(n_ab,n_no)/(n_ab+n_no):.3f}')
    results = {}
    for config in ['baseline','class_weighted','oversampled']:
        per_seed=[]
        for seed in SEEDS:
            tm, preds = train_one(config, seed, tr, va, test_df)
            per_seed.append(tm)
            json.dump(preds, open(f'{OUT_DIR}/preds_{HOLD}_{config}_seed{seed}.json','w'))
            print(f'  {config:15s} s{seed}: acc={tm["accuracy"]:.3f} sens={tm["sensitivity"]:.3f} '
                  f'spec={tm["specificity"]:.3f} f1={tm["f1"]:.3f}')
        results[config]=per_seed
    json.dump(results, open(f'{OUT_DIR}/metrics_{HOLD}.json','w'))

## 5. Within-distribution (pooled random split) trainingAll six databases pooled; stratified random split. This is the comparison design that produces optimistic (leakage-influenced) numbers.

In [ ]:
trainval, test_df = train_test_split(master_df, test_size=0.15, stratify=master_df['label'], random_state=42)
tr, va = train_test_split(trainval, test_size=0.1765, stratify=trainval['label'], random_state=42)
n_ab=int((test_df.label==1).sum()); n_no=int((test_df.label==0).sum())
print(f'WITHIN-DISTRIBUTION | Train {len(tr)} Val {len(va)} Test {len(test_df)} '
      f'| majority-acc={max(n_ab,n_no)/(n_ab+n_no):.3f}')

results = {}
for config in ['baseline','class_weighted','oversampled']:
    per_seed=[]
    for seed in SEEDS:
        tm, preds = train_one(config, seed, tr, va, test_df)
        per_seed.append(tm)
        json.dump(preds, open(f'{OUT_DIR}/preds_within_{config}_seed{seed}.json','w'))
        print(f'  {config:15s} s{seed}: acc={tm["accuracy"]:.3f} sens={tm["sensitivity"]:.3f} '
              f'spec={tm["specificity"]:.3f} f1={tm["f1"]:.3f}')
    results[config]=per_seed
json.dump(results, open(f'{OUT_DIR}/metrics_within.json','w'))

## 6. Confidence intervals (Wilson) and pairwise McNemar testsComputed from saved per-recording predictions; no retraining.

In [ ]:
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar

def load_preds(tag, config, seed):
    f = f'{OUT_DIR}/preds_{tag}_{config}_seed{seed}.json'
    return json.load(open(f)) if os.path.exists(f) else None

CONFIGS=['baseline','class_weighted','oversampled']
TAGS=['within','training-a','training-b','training-e']

print('WILSON 95% CIs (pooled across seeds)')
for tag in TAGS:
    print(f'[{tag}]')
    for cfg in CONFIGS:
        rows=[]
        for s in SEEDS:
            p=load_preds(tag,cfg,s)
            if p: rows += [(yt,yp) for _,yt,yp in p]
        if not rows: continue
        y=np.array([r[0] for r in rows]); pr=np.array([r[1] for r in rows])
        tp=int(((pr==1)&(y==1)).sum()); tn=int(((pr==0)&(y==0)).sum())
        fp=int(((pr==1)&(y==0)).sum()); fn=int(((pr==0)&(y==1)).sum())
        def wil(k,n): return proportion_confint(k,n,alpha=0.05,method='wilson') if n else (float('nan'),)*2
        acc=wil(tp+tn,tp+tn+fp+fn); sens=wil(tp,tp+fn); spec=wil(tn,tn+fp)
        print(f'  {cfg:15s} acc={(tp+tn)/(tp+tn+fp+fn):.3f} [{acc[0]:.3f},{acc[1]:.3f}] '
              f'sens={tp/(tp+fn+1e-9):.3f} [{sens[0]:.3f},{sens[1]:.3f}] '
              f'spec={tn/(tn+fp+1e-9):.3f} [{spec[0]:.3f},{spec[1]:.3f}]')

print('\nPAIRWISE McNEMAR (seed 0; Bonferroni alpha=0.0167)')
for tag in TAGS:
    print(f'[{tag}]')
    preds={c:load_preds(tag,c,0) for c in CONFIGS}
    if any(v is None for v in preds.values()): continue
    maps={c:{rid:(yt,yp) for rid,yt,yp in preds[c]} for c in CONFIGS}
    ids=list(maps[CONFIGS[0]].keys())
    for a,b in [('baseline','class_weighted'),('baseline','oversampled'),('class_weighted','oversampled')]:
        n01=n10=0
        for rid in ids:
            ca = maps[a][rid][1]==maps[a][rid][0]; cb = maps[b][rid][1]==maps[b][rid][0]
            if ca and not cb: n10+=1
            elif cb and not ca: n01+=1
        res=mcnemar([[0,n01],[n10,0]], exact=False, correction=True)
        sig='sig' if res.pvalue<0.0167 else 'ns'
        print(f'  {a:15s} vs {b:15s}: chi2={res.statistic:.3f} p={res.pvalue:.4f} ({sig})')

## 7. Training/validation curves (subset) and plottingRetrains a representative subset (3 models × {within-distribution, held-out training-e} × seed 0) with per-epoch logging, then plots loss and validation sensitivity/specificity. Resumable: skips runs already saved.

In [ ]:
import matplotlib.pyplot as plt
CURVE_DIR=f'{OUT_DIR}/curve_runs'; os.makedirs(CURVE_DIR, exist_ok=True)

def val_metrics(model, loader, crit):
    model.eval(); ys,ps=[],[]; tot=0.0; n=0
    with torch.no_grad():
        for x,y,_ in loader:
            out=model(x.to(DEVICE)); tot+=crit(out,y.to(DEVICE)).item()*len(y); n+=len(y)
            ps+=list(out.argmax(1).cpu().numpy()); ys+=list(y.numpy())
    ys,ps=np.array(ys),np.array(ps); tn,fp,fn,tp=confusion_matrix(ys,ps,labels=[0,1]).ravel(); eps=1e-9
    return tot/n, tp/(tp+fn+eps), tn/(tn+fp+eps)

def train_logged(config, tr, va):
    set_seed(0); g=torch.Generator(); g.manual_seed(0)
    if config=='oversampled':
        c=tr['label'].value_counts().to_dict(); w=tr['label'].map(lambda k:1.0/c[k]).values
        trl=DataLoader(SpecDS(tr),BATCH,sampler=WeightedRandomSampler(torch.DoubleTensor(w),len(w),True,generator=g))
    else: trl=DataLoader(SpecDS(tr),BATCH,shuffle=True,generator=g)
    val=DataLoader(SpecDS(va),BATCH)
    if config=='class_weighted':
        n=tr['label'].value_counts().to_dict(); cw=torch.tensor([1.0/n[0],1.0/n[1]],dtype=torch.float).to(DEVICE); cw=cw/cw.sum()*2
        crit=nn.CrossEntropyLoss(weight=cw)
    else: crit=nn.CrossEntropyLoss()
    cp=nn.CrossEntropyLoss(); model=build_model(); opt=torch.optim.SGD(model.parameters(),lr=LR,momentum=MOMENTUM)
    log={'train_loss':[],'val_loss':[],'val_sens':[],'val_spec':[]}
    for ep in range(EPOCHS):
        model.train(); run=0.0; nn_=0
        for x,y,_ in trl:
            opt.zero_grad(); out=model(x.to(DEVICE)); crit(out,y.to(DEVICE)).backward(); opt.step()
            run+=cp(out.detach(),y.to(DEVICE)).item()*len(y); nn_+=len(y)
        vl,vs,vp=val_metrics(model,val,cp)
        log['train_loss'].append(run/nn_); log['val_loss'].append(vl); log['val_sens'].append(vs); log['val_spec'].append(vp)
    return log

tv,_=train_test_split(master_df,test_size=0.15,stratify=master_df['label'],random_state=42)
w_tr,w_va=train_test_split(tv,test_size=0.1765,stratify=tv['label'],random_state=42)
pool=master_df[master_df['source']!='training-e']
e_tr,e_va=train_test_split(pool,test_size=0.15,stratify=pool['label'],random_state=42)
settings={'Within-distribution':(w_tr,w_va),'Held-out training-e':(e_tr,e_va)}

for sname,(tr,va) in settings.items():
    for cfg in CONFIGS:
        key=f'{sname}|{cfg}'.replace(' ','_')
        fpath=f'{CURVE_DIR}/{key}.json'
        if os.path.exists(fpath): continue
        json.dump(train_logged(cfg,tr,va), open(fpath,'w'))
        print('saved', key)

In [ ]:
# Plot the curves from saved logs
colors={'baseline':'#1f77b4','class_weighted':'#ff7f0e','oversampled':'#2ca02c'}
labels={'baseline':'Baseline','class_weighted':'Class-Weighted','oversampled':'Oversampled'}
logs={}
for s in settings:
    for c in CONFIGS:
        logs[(s,c)]=json.load(open(f"{CURVE_DIR}/{(s+'|'+c).replace(' ','_')}.json"))
ep=range(1,EPOCHS+1)
fig,axes=plt.subplots(2,3,figsize=(13,7))
for ri,s in enumerate(settings):
    ax=axes[ri][0]
    for c in CONFIGS:
        L=logs[(s,c)]
        ax.plot(ep,L['train_loss'],color=colors[c],lw=1.8,label=f'{labels[c]} (train)')
        ax.plot(ep,L['val_loss'],color=colors[c],lw=1.5,ls='--',label=f'{labels[c]} (val)')
    ax.set_title(f'{s}\nLoss'); ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-entropy loss')
    if ri==0: ax.legend(fontsize=6)
    ax=axes[ri][1]
    for c in CONFIGS: ax.plot(ep,logs[(s,c)]['val_sens'],color=colors[c],lw=1.8,label=labels[c])
    ax.set_title(f'{s}\nValidation sensitivity'); ax.set_xlabel('Epoch'); ax.set_ylim(0,1)
    if ri==0: ax.legend(fontsize=7)
    ax=axes[ri][2]
    for c in CONFIGS: ax.plot(ep,logs[(s,c)]['val_spec'],color=colors[c],lw=1.8,label=labels[c])
    ax.set_title(f'{s}\nValidation specificity'); ax.set_xlabel('Epoch'); ax.set_ylim(0,1)
plt.tight_layout()
fig.savefig('/content/drive/MyDrive/ISEF_Project/Figure_curves.png', dpi=300, bbox_inches='tight')
plt.show()